# 1 Импорт библиотек и функций

## 1.1 Библиотеки

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

## 1.2 Функции

In [39]:
def evaluate_classification(y_test, y_pred, y_pred_proba=None):
    """Оценивает результаты бинарной классификации"""
    print(f"\n{'='*50}")
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"Accuracy: {accuracy:.4f}")
    #print(f"Precision: {precision:.4f}")
    #print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    
    #if y_pred_proba is not None:
    #    auc_roc = roc_auc_score(y_test, y_pred_proba)
    #    print(f"AUC-ROC: {auc_roc:.4f}")
    print(f"\n{'='*50}")

def train_predict_rf(X_train, X_test, y_train):
    """Обучает и предсказывает Random Forest для классификации"""
    model = RandomForestClassifier(random_state=13)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    return y_pred, y_pred_proba

def train_predict_catboost(X_train, X_test, y_train):
    """Обучает и предсказывает CatBoost для классификации"""
    model = CatBoostClassifier(verbose=False, random_state=13)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    return y_pred, y_pred_proba

In [5]:
def train_test_split_by_date(df, target_column, test_size=0.2):
    """
    Разбивает данные на train/test по дате и возвращает X, y
    
    Args:
        df: DataFrame с колонкой 'begin'
        target_column: название целевой переменной
        test_size: доля тестовых данных (0.2 = 20%)
    """
    df = df.sort_values('begin').reset_index(drop=True)
    
    # Вычисляем индекс разбиения
    split_idx = int(len(df) * (1 - test_size))
    
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()
    
    print(f"Train: {train_df['begin'].min()} - {train_df['begin'].max()} ({len(train_df)} samples)")
    print(f"Test:  {test_df['begin'].min()} - {test_df['begin'].max()} ({len(test_df)} samples)")
    
    # Удаляем колонку 'begin' и разделяем на X, y
    X_train = train_df.drop(columns=['begin', target_column])
    y_train = train_df[target_column]
    
    X_test = test_df.drop(columns=['begin', target_column])
    y_test = test_df[target_column]
    
    print(f"Признаков: {X_train.shape[1]}")
    
    return X_train, X_test, y_train, y_test

# 2 Подготовка данных

In [72]:
data_SBER = pd.read_csv("../../../data/stock_features_data/stocks_features_SBER.csv")

data_GAZP = pd.read_csv("../../../data/stock_features_data/stocks_features_GAZP.csv")

part_SBER = data_SBER[['begin', 'close', 'target_class']]
data_SBER.drop(['close', 'target_class'], axis=1, inplace=True)

part_GAZP = data_GAZP[['begin', 'close', 'target_class']]
data_GAZP.drop(['close', 'target_class'], axis=1, inplace=True)

data_SBER['target_price_change'] = (data_SBER.target_price_change > 0).astype(int)
data_GAZP['target_price_change'] = (data_GAZP.target_price_change > 0).astype(int)

X_train_SBER, X_test_SBER, y_train_SBER, y_test_SBER = train_test_split_by_date(
    df=data_SBER, 
    target_column='target_price_change',
    test_size=0.2
)

X_train_GAZP, X_test_GAZP, y_train_GAZP, y_test_GAZP = train_test_split_by_date(
    df=data_GAZP, 
    target_column='target_price_change',
    test_size=0.2
)

Train: 2022-07-04 10:00:00 - 2025-03-17 16:00:00 (3342 samples)
Test:  2025-03-17 18:00:00 - 2025-09-30 18:00:00 (836 samples)
Признаков: 83
Train: 2022-07-04 10:00:00 - 2025-03-17 16:00:00 (3342 samples)
Test:  2025-03-17 18:00:00 - 2025-09-30 18:00:00 (836 samples)
Признаков: 83


# 3 Обучение моделей с подбором гиперпараметров

## 3.1 Вот он, лес

### 3.1.1 Сбер

In [77]:
y_pred_SBER, y_proba_SBER = train_predict_rf(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER, y_proba_SBER)


Accuracy: 0.5478
F1-Score: 0.5979



In [41]:
y_pred_SBER, y_proba_SBER = train_predict_rf(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER, y_proba_SBER)


Accuracy: 0.5788
F1-Score: 0.6204



### 3.1.2 Газпром

In [78]:
y_pred_GAZP, y_proba_GAZP = train_predict_rf(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP, y_proba_GAZP)


Accuracy: 0.6459
F1-Score: 0.5647



In [47]:
y_pred_GAZP, y_proba_GAZP = train_predict_rf(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP, y_proba_GAZP)


Accuracy: 0.5881
F1-Score: 0.4165



## 3.2 Бустинг

### 3.2.1 Сбер

In [81]:
y_pred_SBER, y_proba_SBER = train_predict_catboost(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER, y_proba_SBER)


Accuracy: 0.5000
F1-Score: 0.4695



In [54]:
y_pred_SBER, y_proba_SBER = train_predict_catboost(X_train_SBER, X_test_SBER, y_train_SBER)
evaluate_classification(y_test_SBER, y_pred_SBER, y_proba_SBER)


Accuracy: 0.5601
F1-Score: 0.6019



### 3.2.2 Газпром

In [82]:
y_pred_GAZP, y_proba_GAZP = train_predict_catboost(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP, y_proba_GAZP)


Accuracy: 0.6256
F1-Score: 0.5250



In [50]:
y_pred_GAZP, y_proba_GAZP = train_predict_catboost(X_train_GAZP, X_test_GAZP, y_train_GAZP)
evaluate_classification(y_test_GAZP, y_pred_GAZP, y_proba_GAZP)


Accuracy: 0.5764
F1-Score: 0.4319

